In [ ]:
def test_map_utilities():\n    \"\"\"Test map utility functions.\"\"\"\n    # Test HereMapTiles initialization\n    test_api_key = \"test_key_123\"\n    map_tiles = HereMapTiles(test_api_key)\n    \n    assert_true(map_tiles.api_key == test_api_key, \"API key should be stored correctly\")\n    \n    # Test URL generation\n    url_template = map_tiles.get_map_tiles_url()\n    assert_true(\"apiKey=test_key_123\" in url_template, \"API key should be in URL template\")\n    assert_true(\"{z}/{x}/{y}\" in url_template, \"URL should have tile coordinate placeholders\")\n    \n    # Test coordinate conversion functions\n    lat, lon = 52.2053, 0.1218  # Cambridge coordinates\n    zoom = 15\n    \n    # Convert to tile coordinates and back\n    x, y = map_tiles.deg2num(lat, lon, zoom)\n    lat_back, lon_back = map_tiles.num2deg(x, y, zoom)\n    \n    # Should be approximately the same (within tile precision)\n    lat_diff = abs(lat - lat_back)\n    lon_diff = abs(lon - lon_back)\n    \n    assert_true(lat_diff < 0.01, f\"Latitude conversion should be accurate: diff={lat_diff}\")\n    assert_true(lon_diff < 0.01, f\"Longitude conversion should be accurate: diff={lon_diff}\")\n    \n    # Test extent calculation\n    test_lats = [52.2053, 52.2063, 52.2073]\n    test_lons = [0.1218, 0.1228, 0.1238]\n    \n    extent = map_tiles.get_map_extent(test_lats, test_lons, padding=0.001)\n    min_lon, max_lon, min_lat, max_lat = extent\n    \n    assert_true(min_lat < min(test_lats), \"Min lat should include padding\")\n    assert_true(max_lat > max(test_lats), \"Max lat should include padding\")\n    assert_true(min_lon < min(test_lons), \"Min lon should include padding\")\n    assert_true(max_lon > max(test_lons), \"Max lon should include padding\")\n    \n    print(\"  ✅ Map coordinate conversion working\")\n    print(f\"  📍 Test coordinates: ({lat}, {lon}) → tile({x}, {y}) → ({lat_back:.4f}, {lon_back:.4f})\")\n    print(f\"  📏 Extent: {extent}\")\n    return True\n\ndef test_simple_trajectory_plot():\n    \"\"\"Test simple trajectory plotting function.\"\"\"\n    # Test coordinates (small route)\n    test_lats = [52.2053, 52.2060, 52.2070]\n    test_lons = [0.1218, 0.1225, 0.1235]\n    test_speeds = [10.0, 15.0, 20.0]  # m/s\n    \n    try:\n        # Test without speeds\n        fig = create_simple_trajectory_plot(test_lats, test_lons)\n        assert_true(fig is not None, \"Figure should be created\")\n        plt.close(fig)  # Clean up\n        \n        # Test with speeds\n        fig = create_simple_trajectory_plot(test_lats, test_lons, speeds=test_speeds)\n        assert_true(fig is not None, \"Figure with speeds should be created\")\n        plt.close(fig)  # Clean up\n        \n        print(\"  ✅ Trajectory plotting functions working\")\n        print(f\"  📈 Tested with {len(test_lats)} coordinates\")\n        return True\n        \n    except Exception as e:\n        print(f\"  ❌ Trajectory plotting failed: {e}\")\n        return False\n\n# Run map utility tests\nrun_test(\"Map Utilities\", test_map_utilities)\nrun_test(\"Simple Trajectory Plot\", test_simple_trajectory_plot)

# Duty Cycle Prediction Package - Interactive Unit Tests

This notebook provides interactive unit tests for the duty cycle prediction package using real AY71UCD data.

## Table of Contents
1. [Setup and Test Configuration](#setup)
2. [Import Tests](#import-tests)
3. [Basic Functionality Tests](#basic-tests)
4. [Data Processing Tests](#data-tests)
5. [Vehicle Dynamics Tests](#dynamics-tests)
6. [Driving Cycle Generator Tests](#generator-tests)
7. [Edge Cases and Error Handling](#edge-cases)
8. [Performance Tests](#performance)
9. [Integration Tests](#integration)
10. [Test Summary](#summary)

## 1. Setup and Test Configuration {#setup}

In [ ]:
# Standard library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
import warnings
import traceback
warnings.filterwarnings('ignore')

# Add the src directory to the path for development
import sys
sys.path.append('../src')

# Test tracking
test_results = {
    'passed': 0,
    'failed': 0,
    'errors': []
}

def run_test(test_name, test_function):
    """Run a test and track results."""
    try:
        print(f"\n🧪 Testing: {test_name}")
        start_time = time.time()
        result = test_function()
        end_time = time.time()
        
        if result:
            test_results['passed'] += 1
            print(f"✅ PASSED ({end_time - start_time:.3f}s)")
        else:
            test_results['failed'] += 1
            print(f"❌ FAILED ({end_time - start_time:.3f}s)")
        return result
    except Exception as e:
        test_results['failed'] += 1
        test_results['errors'].append(f"{test_name}: {str(e)}")
        print(f"💥 ERROR: {str(e)} ({time.time() - start_time:.3f}s)")
        traceback.print_exc()
        return False

def assert_equal(actual, expected, message="Values should be equal"):
    """Custom assertion function."""
    if actual != expected:
        raise AssertionError(f"{message}. Expected: {expected}, Got: {actual}")

def assert_almost_equal(actual, expected, tolerance=1e-6, message="Values should be almost equal"):
    """Custom assertion for floating point comparison."""
    if abs(actual - expected) > tolerance:
        raise AssertionError(f"{message}. Expected: {expected}, Got: {actual}, Tolerance: {tolerance}")

def assert_true(condition, message="Condition should be True"):
    """Custom assertion for boolean conditions."""
    if not condition:
        raise AssertionError(message)

print("🔧 Test framework initialized")
print(f"📊 Python version: {sys.version}")

def test_package_imports():
    """Test that all package components can be imported."""
    try:
        from duty_cycle_prediction import (
            DrivingCycleGenerator,
            DEFAULT_VEHICLE_PARAMS,
            DEFAULT_DYNAMICS_PARAMS,
            HereMapTiles,
            create_simple_trajectory_plot
        )
        from duty_cycle_prediction.vehicle_dynamics import (
            calculate_wheel_power,
            calculate_fuel_consumption_rate,
            calculate_battery_consumption_rate
        )
        from duty_cycle_prediction.config import (
            PROJECT_ROOT,
            DATA_DIR
        )
        print("  📦 All imports successful")
        print("  🗺️ Map utilities imported successfully")
        return True
    except ImportError as e:
        print(f"  ❌ Import error: {e}")
        return False

In [ ]:
def test_package_imports():
    """Test that all package components can be imported."""
    try:
        from duty_cycle_prediction import (
            DrivingCycleGenerator,
            DEFAULT_VEHICLE_PARAMS,
            DEFAULT_DYNAMICS_PARAMS
        )
        from duty_cycle_prediction.vehicle_dynamics import (
            calculate_wheel_power,
            calculate_fuel_consumption_rate,
            calculate_battery_consumption_rate
        )
        from duty_cycle_prediction.config import (
            PROJECT_ROOT,
            DATA_DIR
        )
        print("  📦 All imports successful")
        return True
    except ImportError as e:
        print(f"  ❌ Import error: {e}")
        return False

def test_dependencies():
    """Test that all dependencies are available."""
    try:
        import pandas
        import numpy
        import matplotlib
        import scipy
        print(f"  📊 pandas: {pandas.__version__}")
        print(f"  🔢 numpy: {numpy.__version__}")
        print(f"  📈 matplotlib: {matplotlib.__version__}")
        print(f"  🧮 scipy: {scipy.__version__}")
        return True
    except ImportError as e:
        print(f"  ❌ Dependency error: {e}")
        return False

# Run import tests
run_test("Package Imports", test_package_imports)
run_test("Dependencies", test_dependencies)

In [ ]:
# Import the package after confirming imports work
from duty_cycle_prediction import (
    DrivingCycleGenerator,
    DEFAULT_VEHICLE_PARAMS,
    DEFAULT_DYNAMICS_PARAMS,
    HereMapTiles,
    create_simple_trajectory_plot
)
from duty_cycle_prediction.vehicle_dynamics import (
    calculate_wheel_power,
    calculate_fuel_consumption_rate,
    calculate_battery_consumption_rate
)
from duty_cycle_prediction.config import (
    PROJECT_ROOT,
    DATA_DIR
)

print("✅ All imports completed successfully")
print("🗺️ Map utilities available for testing")

## 3. Basic Functionality Tests {#basic-tests}

In [ ]:
def test_driving_cycle_generator_initialization():
    """Test DrivingCycleGenerator can be initialized."""
    generator = DrivingCycleGenerator()
    assert_true(generator is not None, "Generator should be created")
    print("  ✅ Generator initialized successfully")
    return True

def test_haversine_distance():
    """Test haversine distance calculation with known values."""
    # London to Cambridge (approximately 87.5 km)
    lat1, lon1 = 51.5074, -0.1278  # London
    lat2, lon2 = 52.2053, 0.1218   # Cambridge
    
    distance = DrivingCycleGenerator.haversine_distance(lat1, lon1, lat2, lon2)
    
    # Should be approximately 87,500 meters (within 2000m tolerance)
    expected_distance = 87500
    tolerance = 2000
    
    assert_true(abs(distance - expected_distance) < tolerance, 
                f"Distance should be ~{expected_distance}m, got {distance:.0f}m")
    print(f"  ✅ Distance calculated: {distance:.0f}m (expected ~{expected_distance}m)")
    return True

def test_speed_interpolation():
    """Test speed interpolation function."""
    # Test linear interpolation
    result = DrivingCycleGenerator.interpolate_speed(10.0, 20.0, 0.5)
    assert_almost_equal(result, 15.0, message="50% interpolation should give midpoint")
    
    # Test edge cases
    result = DrivingCycleGenerator.interpolate_speed(0.0, 10.0, 0.0)
    assert_almost_equal(result, 0.0, message="0% interpolation should give start value")
    
    result = DrivingCycleGenerator.interpolate_speed(0.0, 10.0, 1.0)
    assert_almost_equal(result, 10.0, message="100% interpolation should give end value")
    
    print("  ✅ Speed interpolation working correctly")
    return True

def test_speed_update():
    """Test speed update function."""
    # Test acceleration
    v_new = DrivingCycleGenerator._update_speed(
        v_cur=10.0, v_desired=15.0, a_acc=2.0, a_dec=2.0, dt=1.0
    )
    assert_almost_equal(v_new, 12.0, message="Acceleration should increase speed by a_acc*dt")
    
    # Test deceleration
    v_new = DrivingCycleGenerator._update_speed(
        v_cur=15.0, v_desired=10.0, a_acc=2.0, a_dec=2.0, dt=1.0
    )
    assert_almost_equal(v_new, 13.0, message="Deceleration should decrease speed by a_dec*dt")
    
    # Test speed limiting
    v_new = DrivingCycleGenerator._update_speed(
        v_cur=14.0, v_desired=15.0, a_acc=2.0, a_dec=2.0, dt=1.0
    )
    assert_almost_equal(v_new, 15.0, message="Speed should not exceed desired speed")
    
    print("  ✅ Speed update function working correctly")
    return True

# Run basic functionality tests
run_test("Generator Initialization", test_driving_cycle_generator_initialization)
run_test("Haversine Distance", test_haversine_distance)
run_test("Speed Interpolation", test_speed_interpolation)
run_test("Speed Update", test_speed_update)

## 4. Data Processing Tests {#data-tests}

In [ ]:
def test_data_loading():
    """Test loading AY71UCD data."""
    data_file = '../data/AY71UCD/20250227_AY71UCD_Leg1.csv'
    
    try:
        data = pd.read_csv(data_file)
        print(f"  📊 Loaded {len(data)} rows from {data_file}")
        
        # Check required columns exist
        required_cols = ['Longitude', 'Latitude', 'Spd_Kmph_x']
        for col in required_cols:
            assert_true(col in data.columns, f"Column '{col}' should exist")
        
        # Check data types
        assert_true(len(data) > 0, "Data should not be empty")
        assert_true(not data['Longitude'].isna().all(), "Longitude should have valid values")
        assert_true(not data['Latitude'].isna().all(), "Latitude should have valid values")
        
        print(f"  ✅ Data validation passed")
        return True
    except Exception as e:
        print(f"  ❌ Data loading failed: {e}")
        return False

def test_route_data_preparation():
    """Test converting AY71UCD data to route format."""
    # Load sample data
    data_file = '../data/AY71UCD/20250227_AY71UCD_Leg1.csv'
    raw_data = pd.read_csv(data_file)
    
    # Take a small sample for testing
    sample_data = raw_data.iloc[:100:10].copy()  # Every 10th row, first 100 rows
    
    # Create route DataFrame
    route_df = pd.DataFrame({
        'Lat': sample_data['Latitude'],
        'Lon': sample_data['Longitude'],
        'MaxSpeed': 25.0,
        'BaseSpeed': 25.0,
        'TrafficSpeed': sample_data['Spd_Kmph_x'] / 3.6,  # Convert to m/s
    })
    
    # Add actions
    actions = ['start'] + ['continue'] * (len(route_df) - 2) + ['arrive']
    route_df['Action'] = actions
    
    # Clean data
    route_df = route_df.dropna().reset_index(drop=True)
    route_df['TrafficSpeed'] = route_df['TrafficSpeed'].clip(0, 30)
    
    # Validate route data
    assert_true(len(route_df) >= 3, "Route should have at least 3 points")
    assert_true('Lat' in route_df.columns, "Route should have Lat column")
    assert_true('Lon' in route_df.columns, "Route should have Lon column")
    assert_true('Action' in route_df.columns, "Route should have Action column")
    assert_true(route_df.iloc[0]['Action'] == 'start', "First action should be 'start'")
    assert_true(route_df.iloc[-1]['Action'] == 'arrive', "Last action should be 'arrive'")
    
    print(f"  ✅ Route data prepared: {len(route_df)} waypoints")
    return True

def test_configuration_parameters():
    """Test that configuration parameters are valid."""
    # Test vehicle parameters
    assert_true(isinstance(DEFAULT_VEHICLE_PARAMS, dict), "Vehicle params should be a dictionary")
    assert_true('v_cap' in DEFAULT_VEHICLE_PARAMS, "Should have v_cap parameter")
    assert_true(DEFAULT_VEHICLE_PARAMS['v_cap'] > 0, "v_cap should be positive")
    
    # Test dynamics parameters
    assert_true(isinstance(DEFAULT_DYNAMICS_PARAMS, dict), "Dynamics params should be a dictionary")
    assert_true('mass_kg' in DEFAULT_DYNAMICS_PARAMS, "Should have mass_kg parameter")
    assert_true(DEFAULT_DYNAMICS_PARAMS['mass_kg'] > 0, "mass_kg should be positive")
    
    print(f"  ✅ Configuration parameters valid")
    print(f"    Vehicle params: {len(DEFAULT_VEHICLE_PARAMS)} items")
    print(f"    Dynamics params: {len(DEFAULT_DYNAMICS_PARAMS)} items")
    return True

# Run data processing tests
run_test("Data Loading", test_data_loading)
run_test("Route Data Preparation", test_route_data_preparation)
run_test("Configuration Parameters", test_configuration_parameters)

## 5. Vehicle Dynamics Tests {#dynamics-tests}

In [ ]:
def test_wheel_power_calculation():
    """Test wheel power calculation with known scenarios."""
    # Test case 1: Stationary vehicle
    power = calculate_wheel_power(
        mass_kg=5000.0,
        gradient_degrees=0.0,
        velocity_mps=0.0,
        acceleration_mps2=0.0
    )
    assert_almost_equal(power, 0.0, message="Stationary vehicle should require 0 power")
    
    # Test case 2: Constant speed on flat road
    power = calculate_wheel_power(
        mass_kg=5000.0,
        gradient_degrees=0.0,
        velocity_mps=20.0,
        acceleration_mps2=0.0
    )
    assert_true(power > 0, "Moving vehicle should require positive power")
    
    # Test case 3: Uphill vs flat
    power_flat = calculate_wheel_power(
        mass_kg=5000.0, gradient_degrees=0.0, velocity_mps=20.0, acceleration_mps2=0.0
    )
    power_uphill = calculate_wheel_power(
        mass_kg=5000.0, gradient_degrees=5.0, velocity_mps=20.0, acceleration_mps2=0.0
    )
    assert_true(power_uphill > power_flat, "Uphill should require more power")
    
    # Test case 4: With force components
    power, components = calculate_wheel_power(
        mass_kg=5000.0,
        gradient_degrees=0.0,
        velocity_mps=20.0,
        acceleration_mps2=0.0,
        return_components=True
    )
    
    assert_true(isinstance(components, dict), "Components should be a dictionary")
    required_components = ['rolling', 'grade', 'aerodynamic', 'acceleration', 'total']
    for comp in required_components:
        assert_true(comp in components, f"Should have {comp} component")
    
    # On flat road with no acceleration, grade and acceleration forces should be ~0
    assert_almost_equal(components['grade'], 0.0, tolerance=1e-6, 
                        message="Grade force should be 0 on flat road")
    assert_almost_equal(components['acceleration'], 0.0, tolerance=1e-6,
                        message="Acceleration force should be 0 at constant speed")
    
    print(f"  ✅ Power calculations validated")
    print(f"    Flat road power: {power_flat/1000:.1f} kW")
    print(f"    Uphill power: {power_uphill/1000:.1f} kW")
    return True

def test_fuel_consumption_calculation():
    """Test fuel consumption calculation."""
    # Test positive power (driving)
    result = calculate_fuel_consumption_rate(wheel_power_watts=50000.0)
    
    assert_true(isinstance(result, dict), "Result should be a dictionary")
    assert_true('rate' in result, "Result should have 'rate' key")
    assert_true('unit' in result, "Result should have 'unit' key")
    assert_equal(result['unit'], 'L/hr', "Unit should be L/hr")
    assert_true(result['rate'] > 0, "Fuel rate should be positive for positive power")
    
    # Test negative power (braking)
    result_negative = calculate_fuel_consumption_rate(wheel_power_watts=-10000.0)
    assert_true(result_negative['rate'] >= 0, "Fuel rate should not be negative (idle consumption)")
    
    # Test zero power
    result_zero = calculate_fuel_consumption_rate(wheel_power_watts=0.0)
    assert_true(result_zero['rate'] >= 0, "Fuel rate should not be negative at zero power")
    
    print(f"  ✅ Fuel consumption calculations validated")
    print(f"    50kW power → {result['rate']:.2f} L/hr")
    return True

def test_battery_consumption_calculation():
    """Test battery consumption calculation for electric vehicles."""
    # Test positive power (driving)
    result = calculate_battery_consumption_rate(wheel_power_watts=50000.0)
    
    assert_true(isinstance(result, dict), "Result should be a dictionary")
    assert_true('rate' in result, "Result should have 'rate' key")
    assert_true('unit' in result, "Result should have 'unit' key")
    assert_equal(result['unit'], 'kW', "Unit should be kW")
    assert_true(result['rate'] > 0, "Battery rate should be positive for positive power")
    
    # Test negative power (regenerative braking)
    result_negative = calculate_battery_consumption_rate(wheel_power_watts=-10000.0)
    assert_true(result_negative['rate'] < 0, "Battery rate should be negative for regenerative braking")
    
    # Test efficiency
    wheel_power_kw = 50.0
    efficiency = DEFAULT_DYNAMICS_PARAMS['battery_to_wheel_efficiency']
    expected_battery_power = wheel_power_kw / efficiency
    
    result = calculate_battery_consumption_rate(wheel_power_watts=wheel_power_kw * 1000)
    assert_almost_equal(result['rate'], expected_battery_power, tolerance=0.1,
                        message="Battery power should account for efficiency")
    
    print(f"  ✅ Battery consumption calculations validated")
    print(f"    50kW wheel power → {result['rate']:.1f} kW battery power")
    print(f"    Efficiency: {efficiency*100:.1f}%")
    return True

# Run vehicle dynamics tests
run_test("Wheel Power Calculation", test_wheel_power_calculation)
run_test("Fuel Consumption Calculation", test_fuel_consumption_calculation)
run_test("Battery Consumption Calculation", test_battery_consumption_calculation)

## 6. Driving Cycle Generator Tests {#generator-tests}

In [ ]:
def test_simple_driving_cycle():
    """Test generating a simple driving cycle."""
    # Create minimal route data
    route_data = {
        'Lat': [52.2053, 52.2060, 52.2070],
        'Lon': [0.1218, 0.1225, 0.1235],
        'MaxSpeed': [25.0, 25.0, 25.0],
        'BaseSpeed': [25.0, 25.0, 25.0],
        'TrafficSpeed': [20.0, 20.0, 20.0],
        'Action': ['start', 'continue', 'arrive']
    }
    route_df = pd.DataFrame(route_data)
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    # Generate driving cycle with minimal parameters
    result = generator.create_driving_cycle(
        route_df=route_df,
        start_time=start_time,
        dt=1.0,
        smooth_speed=False  # Disable smoothing for simpler test
    )
    
    # Validate result
    assert_true(isinstance(result, pd.DataFrame), "Result should be a DataFrame")
    assert_true(len(result) > 0, "Result should not be empty")
    
    required_columns = ['timestamp', 'Lat', 'Lon', 'distance', 'speed', 'speed_desired', 'acc']
    for col in required_columns:
        assert_true(col in result.columns, f"Result should have '{col}' column")
    
    # Check data types and ranges
    assert_true(result['speed'].min() >= 0, "Speed should not be negative")
    assert_true(result['distance'].is_monotonic_increasing, "Distance should be monotonic increasing")
    
    print(f"  ✅ Simple driving cycle generated: {len(result)} time steps")
    print(f"    Duration: {len(result)} seconds")
    print(f"    Distance: {result['distance'].iloc[-1]:.0f} meters")
    return True

def test_real_data_driving_cycle():
    """Test generating driving cycle with real AY71UCD data."""
    # Load and prepare real data
    data_file = '../data/AY71UCD/20250227_AY71UCD_Leg1.csv'
    raw_data = pd.read_csv(data_file)
    
    # Take a small sample for testing (every 100th point, first 1000 points)
    sample_data = raw_data.iloc[:1000:100].copy()
    
    # Prepare route data
    route_df = pd.DataFrame({
        'Lat': sample_data['Latitude'],
        'Lon': sample_data['Longitude'],
        'MaxSpeed': 25.0,
        'BaseSpeed': 25.0,
        'TrafficSpeed': sample_data['Spd_Kmph_x'] / 3.6,  # Convert to m/s
    })
    
    # Add actions
    actions = ['start'] + ['continue'] * (len(route_df) - 2) + ['arrive']
    route_df['Action'] = actions
    
    # Clean data
    route_df = route_df.dropna().reset_index(drop=True)
    route_df['TrafficSpeed'] = route_df['TrafficSpeed'].clip(0, 30)
    
    # Ensure we have enough data points
    if len(route_df) < 3:
        print("  ⚠️ Insufficient data points for test")
        return True  # Skip test if not enough data
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2025, 2, 27, 9, 0, 0)
    
    # Generate driving cycle
    result = generator.create_driving_cycle(
        route_df=route_df,
        start_time=start_time,
        **DEFAULT_VEHICLE_PARAMS
    )
    
    # Validate result
    assert_true(isinstance(result, pd.DataFrame), "Result should be a DataFrame")
    assert_true(len(result) > 0, "Result should not be empty")
    
    # Check realistic values
    max_speed_kmh = result['speed'].max() * 3.6
    assert_true(max_speed_kmh <= 100, f"Max speed should be reasonable: {max_speed_kmh:.1f} km/h")
    
    avg_speed_kmh = result['speed'].mean() * 3.6
    assert_true(0 <= avg_speed_kmh <= 100, f"Average speed should be reasonable: {avg_speed_kmh:.1f} km/h")
    
    print(f"  ✅ Real data driving cycle generated: {len(result)} time steps")
    print(f"    Average speed: {avg_speed_kmh:.1f} km/h")
    print(f"    Max speed: {max_speed_kmh:.1f} km/h")
    print(f"    Total distance: {result['distance'].iloc[-1]/1000:.2f} km")
    return True

def test_parameter_variations():
    """Test driving cycle generation with different parameters."""
    # Simple route for testing
    route_data = {
        'Lat': [52.2053, 52.2060, 52.2070, 52.2080],
        'Lon': [0.1218, 0.1225, 0.1235, 0.1245],
        'MaxSpeed': [25.0, 25.0, 25.0, 25.0],
        'BaseSpeed': [25.0, 25.0, 25.0, 25.0],
        'TrafficSpeed': [20.0, 20.0, 20.0, 20.0],
        'Action': ['start', 'continue', 'continue', 'arrive']
    }
    route_df = pd.DataFrame(route_data)
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    # Test different acceleration parameters
    test_params = [
        {'a_acc': 0.3, 'a_dec': 0.5},  # Conservative
        {'a_acc': 0.8, 'a_dec': 1.0},  # Aggressive
        {'v_cruise': 15.0},             # Lower cruise speed
        {'dt': 0.5}                     # Smaller time step
    ]
    
    for i, params in enumerate(test_params):
        test_params_full = DEFAULT_VEHICLE_PARAMS.copy()
        test_params_full.update(params)
        
        result = generator.create_driving_cycle(
            route_df=route_df,
            start_time=start_time,
            **test_params_full
        )
        
        assert_true(len(result) > 0, f"Parameter set {i+1} should produce valid result")
        assert_true(result['speed'].min() >= 0, f"Parameter set {i+1}: Speed should not be negative")
        
        print(f"    Test {i+1}: {params} → {len(result)} steps, max speed {result['speed'].max()*3.6:.1f} km/h")
    
    print(f"  ✅ Parameter variation tests passed")
    return True

# Run driving cycle generator tests
run_test("Simple Driving Cycle", test_simple_driving_cycle)
run_test("Real Data Driving Cycle", test_real_data_driving_cycle)
run_test("Parameter Variations", test_parameter_variations)

## 7. Edge Cases and Error Handling {#edge-cases}

In [ ]:
def test_empty_route_data():
    """Test handling of empty route data."""
    empty_df = pd.DataFrame()
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    try:
        result = generator.create_driving_cycle(
            route_df=empty_df,
            start_time=start_time
        )
        # Should not reach here
        return False
    except ValueError as e:
        print(f"  ✅ Correctly caught ValueError for empty data: {e}")
        return True
    except Exception as e:
        print(f"  ⚠️ Unexpected exception type: {type(e).__name__}: {e}")
        return True  # Still acceptable as long as it fails gracefully

def test_insufficient_route_data():
    """Test handling of insufficient route data (less than 2 points)."""
    single_point_df = pd.DataFrame({
        'Lat': [52.2053],
        'Lon': [0.1218],
        'MaxSpeed': [25.0],
        'BaseSpeed': [25.0],
        'TrafficSpeed': [20.0],
        'Action': ['start']
    })
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    try:
        result = generator.create_driving_cycle(
            route_df=single_point_df,
            start_time=start_time
        )
        return False  # Should have failed
    except ValueError as e:
        print(f"  ✅ Correctly caught ValueError for insufficient data: {e}")
        return True
    except Exception as e:
        print(f"  ⚠️ Unexpected exception type: {type(e).__name__}: {e}")
        return True

def test_invalid_coordinates():
    """Test handling of invalid coordinates."""
    invalid_coords_df = pd.DataFrame({
        'Lat': [91.0, 52.2060, 52.2070],  # Invalid latitude > 90
        'Lon': [0.1218, 181.0, 0.1235],   # Invalid longitude > 180
        'MaxSpeed': [25.0, 25.0, 25.0],
        'BaseSpeed': [25.0, 25.0, 25.0],
        'TrafficSpeed': [20.0, 20.0, 20.0],
        'Action': ['start', 'continue', 'arrive']
    })
    
    # Test haversine with invalid coordinates
    try:
        # This should still work but may give unrealistic results
        distance = DrivingCycleGenerator.haversine_distance(91.0, 0.0, 52.0, 0.0)
        print(f"  ⚠️ Invalid coordinates processed (distance: {distance:.0f}m)")
        return True  # Function handles it, even if results are unrealistic
    except Exception as e:
        print(f"  ✅ Exception caught for invalid coordinates: {e}")
        return True

def test_negative_vehicle_parameters():
    """Test handling of negative vehicle parameters."""
    route_data = {
        'Lat': [52.2053, 52.2060, 52.2070],
        'Lon': [0.1218, 0.1225, 0.1235],
        'MaxSpeed': [25.0, 25.0, 25.0],
        'BaseSpeed': [25.0, 25.0, 25.0],
        'TrafficSpeed': [20.0, 20.0, 20.0],
        'Action': ['start', 'continue', 'arrive']
    }
    route_df = pd.DataFrame(route_data)
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    # Test with negative acceleration
    try:
        result = generator.create_driving_cycle(
            route_df=route_df,
            start_time=start_time,
            a_acc=-0.5,  # Negative acceleration
            a_dec=-0.5   # Negative deceleration
        )
        # This might work but produce strange results
        print(f"  ⚠️ Negative parameters processed (may produce unexpected results)")
        return True
    except Exception as e:
        print(f"  ✅ Exception caught for negative parameters: {e}")
        return True

def test_extreme_power_values():
    """Test vehicle dynamics with extreme power values."""
    # Test very high power
    high_power = calculate_wheel_power(
        mass_kg=50000.0,  # Very heavy vehicle
        gradient_degrees=20.0,  # Very steep hill
        velocity_mps=50.0,  # Very high speed
        acceleration_mps2=5.0  # Very high acceleration
    )
    assert_true(high_power > 0, "High power scenario should give positive result")
    print(f"    Extreme scenario power: {high_power/1000:.0f} kW")
    
    # Test fuel consumption with extreme power
    fuel_result = calculate_fuel_consumption_rate(high_power)
    assert_true(fuel_result['rate'] > 0, "Extreme power should give positive fuel consumption")
    print(f"    Extreme fuel consumption: {fuel_result['rate']:.1f} L/hr")
    
    # Test zero mass (should handle gracefully or fail appropriately)
    try:
        zero_mass_power = calculate_wheel_power(
            mass_kg=0.0,
            gradient_degrees=0.0,
            velocity_mps=20.0,
            acceleration_mps2=1.0
        )
        print(f"    Zero mass handled: {zero_mass_power:.1f} W")
    except Exception as e:
        print(f"    Zero mass exception (expected): {e}")
    
    print(f"  ✅ Extreme value tests completed")
    return True

# Run edge case tests
run_test("Empty Route Data", test_empty_route_data)
run_test("Insufficient Route Data", test_insufficient_route_data)
run_test("Invalid Coordinates", test_invalid_coordinates)
run_test("Negative Vehicle Parameters", test_negative_vehicle_parameters)
run_test("Extreme Power Values", test_extreme_power_values)

## 8. Performance Tests {#performance}

In [ ]:
def test_performance_large_dataset():
    """Test performance with a larger dataset."""
    # Load real data
    data_file = '../data/AY71UCD/20250227_AY71UCD_Leg1.csv'
    raw_data = pd.read_csv(data_file)
    
    # Use a moderate sample size for testing
    sample_size = min(500, len(raw_data) // 10)  # Every 10th point, max 500 points
    sample_data = raw_data.iloc[:sample_size*10:10].copy()
    
    # Prepare route data
    route_df = pd.DataFrame({
        'Lat': sample_data['Latitude'],
        'Lon': sample_data['Longitude'],
        'MaxSpeed': 25.0,
        'BaseSpeed': 25.0,
        'TrafficSpeed': sample_data['Spd_Kmph_x'] / 3.6,
    })
    
    actions = ['start'] + ['continue'] * (len(route_df) - 2) + ['arrive']
    route_df['Action'] = actions
    route_df = route_df.dropna().reset_index(drop=True)
    route_df['TrafficSpeed'] = route_df['TrafficSpeed'].clip(0, 30)
    
    if len(route_df) < 10:
        print(f"  ⚠️ Insufficient data for performance test ({len(route_df)} points)")
        return True
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2025, 2, 27, 9, 0, 0)
    
    # Measure generation time
    start = time.time()
    result = generator.create_driving_cycle(
        route_df=route_df,
        start_time=start_time,
        **DEFAULT_VEHICLE_PARAMS
    )
    generation_time = time.time() - start
    
    # Measure dynamics calculation time
    start = time.time()
    for idx, row in result.head(100).iterrows():  # Test first 100 rows
        power = calculate_wheel_power(
            mass_kg=5000.0,
            gradient_degrees=0.0,
            velocity_mps=row['speed'],
            acceleration_mps2=row['acc']
        )
        fuel_rate = calculate_fuel_consumption_rate(power)
    dynamics_time = time.time() - start
    
    # Calculate performance metrics
    waypoints_per_sec = len(route_df) / generation_time
    timesteps_per_sec = len(result) / generation_time
    dynamics_per_sec = 100 / dynamics_time
    
    print(f"  📊 Performance Results:")
    print(f"    Route points: {len(route_df)}")
    print(f"    Generated time steps: {len(result)}")
    print(f"    Generation time: {generation_time:.2f}s")
    print(f"    Processing rate: {waypoints_per_sec:.1f} waypoints/s, {timesteps_per_sec:.1f} timesteps/s")
    print(f"    Dynamics calculation: {dynamics_per_sec:.1f} calculations/s")
    
    # Performance assertions
    assert_true(generation_time < 60, "Generation should complete within 60 seconds")
    assert_true(waypoints_per_sec > 1, "Should process at least 1 waypoint per second")
    assert_true(dynamics_per_sec > 10, "Should perform at least 10 dynamics calculations per second")
    
    print(f"  ✅ Performance tests passed")
    return True

def test_memory_usage():
    """Test memory usage with different dataset sizes."""
    import psutil
    import os
    
    process = psutil.Process(os.getpid())
    initial_memory = process.memory_info().rss / 1024 / 1024  # MB
    
    # Test with increasing dataset sizes
    sizes = [10, 50, 100, 200]
    memory_usage = []
    
    for size in sizes:
        # Create synthetic route data
        lats = np.linspace(52.2053, 52.2253, size)  # 2km route
        lons = np.linspace(0.1218, 0.1418, size)
        
        route_df = pd.DataFrame({
            'Lat': lats,
            'Lon': lons,
            'MaxSpeed': 25.0,
            'BaseSpeed': 25.0,
            'TrafficSpeed': 20.0,
        })
        
        actions = ['start'] + ['continue'] * (size - 2) + ['arrive']
        route_df['Action'] = actions
        
        generator = DrivingCycleGenerator()
        start_time = datetime(2023, 1, 1, 10, 0, 0)
        
        # Generate driving cycle
        result = generator.create_driving_cycle(
            route_df=route_df,
            start_time=start_time,
            dt=1.0,
            smooth_speed=False  # Disable to reduce processing
        )
        
        current_memory = process.memory_info().rss / 1024 / 1024  # MB
        memory_usage.append(current_memory - initial_memory)
        
        print(f"    {size} waypoints → {len(result)} timesteps, Memory: +{memory_usage[-1]:.1f}MB")
    
    # Check memory growth is reasonable
    max_memory_increase = max(memory_usage)
    assert_true(max_memory_increase < 500, f"Memory usage should be reasonable: {max_memory_increase:.1f}MB")
    
    print(f"  ✅ Memory usage test passed (max increase: {max_memory_increase:.1f}MB)")
    return True

# Run performance tests
run_test("Performance Large Dataset", test_performance_large_dataset)
run_test("Memory Usage", test_memory_usage)

## 9. Integration Tests {#integration}

In [ ]:
def test_complete_workflow():
    """Test the complete workflow from data loading to analysis."""
    print("  🔄 Running complete workflow test...")
    
    # Step 1: Load data
    data_file = '../data/AY71UCD/20250227_AY71UCD_Leg1.csv'
    raw_data = pd.read_csv(data_file)
    print(f"    1. Loaded {len(raw_data)} data points")
    
    # Step 2: Prepare route
    sample_data = raw_data.iloc[:2000:50].copy()  # Every 50th point from first 2000
    route_df = pd.DataFrame({
        'Lat': sample_data['Latitude'],
        'Lon': sample_data['Longitude'],
        'MaxSpeed': 25.0,
        'BaseSpeed': 25.0,
        'TrafficSpeed': sample_data['Spd_Kmph_x'] / 3.6,
    })
    actions = ['start'] + ['continue'] * (len(route_df) - 2) + ['arrive']
    route_df['Action'] = actions
    route_df = route_df.dropna().reset_index(drop=True)
    route_df['TrafficSpeed'] = route_df['TrafficSpeed'].clip(0, 30)
    print(f"    2. Prepared route with {len(route_df)} waypoints")
    
    # Step 3: Generate driving cycle
    generator = DrivingCycleGenerator()
    start_time = datetime(2025, 2, 27, 9, 0, 0)
    
    driving_cycle = generator.create_driving_cycle(
        route_df=route_df,
        start_time=start_time,
        **DEFAULT_VEHICLE_PARAMS
    )
    print(f"    3. Generated driving cycle with {len(driving_cycle)} time steps")
    
    # Step 4: Calculate vehicle dynamics
    mass_kg = DEFAULT_DYNAMICS_PARAMS['mass_kg']
    powers = []
    fuel_rates = []
    
    for idx, row in driving_cycle.iterrows():
        power = calculate_wheel_power(
            mass_kg=mass_kg,
            gradient_degrees=0.0,
            velocity_mps=row['speed'],
            acceleration_mps2=row['acc']
        )
        powers.append(power)
        
        fuel_result = calculate_fuel_consumption_rate(power)
        fuel_rates.append(fuel_result['rate'])
    
    driving_cycle['power_kw'] = np.array(powers) / 1000
    driving_cycle['fuel_rate_l_hr'] = fuel_rates
    print(f"    4. Calculated dynamics for all time steps")
    
    # Step 5: Analyze results
    total_distance_km = driving_cycle['distance'].iloc[-1] / 1000
    total_time_hr = len(driving_cycle) / 3600
    avg_speed_kmh = driving_cycle['speed'].mean() * 3.6
    max_power_kw = driving_cycle['power_kw'].max()
    avg_fuel_rate = driving_cycle['fuel_rate_l_hr'].mean()
    total_fuel_l = avg_fuel_rate * total_time_hr
    
    print(f"    5. Analysis complete:")
    print(f"       Distance: {total_distance_km:.2f} km")
    print(f"       Duration: {total_time_hr*60:.1f} minutes")
    print(f"       Average speed: {avg_speed_kmh:.1f} km/h")
    print(f"       Max power: {max_power_kw:.1f} kW")
    print(f"       Total fuel: {total_fuel_l:.2f} L")
    print(f"       Fuel efficiency: {total_distance_km/total_fuel_l:.2f} km/L")
    
    # Validate results
    assert_true(total_distance_km > 0, "Should have traveled some distance")
    assert_true(avg_speed_kmh > 0, "Should have positive average speed")
    assert_true(max_power_kw >= 0, "Power should be non-negative")
    assert_true(total_fuel_l > 0, "Should have consumed some fuel")
    assert_true(1 <= total_distance_km/total_fuel_l <= 20, "Fuel efficiency should be realistic")
    
    print(f"  ✅ Complete workflow test passed")
    return True

def test_data_consistency():
    """Test data consistency across multiple runs."""
    # Simple deterministic route
    route_data = {
        'Lat': [52.2053, 52.2063, 52.2073, 52.2083],
        'Lon': [0.1218, 0.1228, 0.1238, 0.1248],
        'MaxSpeed': [25.0, 25.0, 25.0, 25.0],
        'BaseSpeed': [25.0, 25.0, 25.0, 25.0],
        'TrafficSpeed': [20.0, 20.0, 20.0, 20.0],
        'Action': ['start', 'continue', 'continue', 'arrive']
    }
    route_df = pd.DataFrame(route_data)
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    # Run multiple times with same parameters
    results = []
    for i in range(3):
        result = generator.create_driving_cycle(
            route_df=route_df,
            start_time=start_time,
            dt=1.0,
            smooth_speed=False,  # Disable for deterministic results
            **DEFAULT_VEHICLE_PARAMS
        )
        results.append(result)
    
    # Check consistency
    for i in range(1, len(results)):
        # Check same number of time steps
        assert_equal(len(results[i]), len(results[0]), 
                    f"Run {i+1} should have same length as run 1")
        
        # Check speed profiles are identical
        speed_diff = np.abs(results[i]['speed'].values - results[0]['speed'].values).max()
        assert_true(speed_diff < 1e-10, 
                   f"Run {i+1} speed profile should be identical to run 1 (max diff: {speed_diff})")
        
        # Check final distance is identical
        dist_diff = abs(results[i]['distance'].iloc[-1] - results[0]['distance'].iloc[-1])
        assert_true(dist_diff < 1e-6, 
                   f"Run {i+1} final distance should match run 1 (diff: {dist_diff}m)")
    
    print(f"  ✅ Data consistency test passed ({len(results)} runs)")
    print(f"    All runs produced {len(results[0])} time steps")
    print(f"    Final distance: {results[0]['distance'].iloc[-1]:.2f}m")
    return True

def test_configuration_integration():
    """Test integration with different configurations."""
    route_data = {
        'Lat': [52.2053, 52.2063, 52.2073],
        'Lon': [0.1218, 0.1228, 0.1238],
        'MaxSpeed': [25.0, 25.0, 25.0],
        'BaseSpeed': [25.0, 25.0, 25.0],
        'TrafficSpeed': [20.0, 20.0, 20.0],
        'Action': ['start', 'continue', 'arrive']
    }
    route_df = pd.DataFrame(route_data)
    
    generator = DrivingCycleGenerator()
    start_time = datetime(2023, 1, 1, 10, 0, 0)
    
    # Test with default configurations
    result_default_vehicle = generator.create_driving_cycle(
        route_df=route_df, start_time=start_time, **DEFAULT_VEHICLE_PARAMS
    )
    
    # Test dynamics with default configuration
    sample_power = calculate_wheel_power(
        velocity_mps=20.0, acceleration_mps2=0.5, 
        **{k: v for k, v in DEFAULT_DYNAMICS_PARAMS.items() 
           if k in ['mass_kg', 'rolling_resistance_coeff', 'drag_coefficient', 
                   'frontal_area_m2', 'air_density_kg_m3', 'g']},
        gradient_degrees=0.0
    )
    
    sample_fuel = calculate_fuel_consumption_rate(
        sample_power,
        **{k: v for k, v in DEFAULT_DYNAMICS_PARAMS.items() 
           if k in ['powertrain_efficiency', 'engine_efficiency', 
                   'heating_value_mj_l', 'idle_fuel_consumption_l_per_hr']}
    )
    
    sample_battery = calculate_battery_consumption_rate(
        sample_power,
        battery_to_wheel_efficiency=DEFAULT_DYNAMICS_PARAMS['battery_to_wheel_efficiency']
    )
    
    # Validate integration
    assert_true(len(result_default_vehicle) > 0, "Default vehicle config should work")
    assert_true(sample_power > 0, "Default dynamics config should work")
    assert_true(sample_fuel['rate'] > 0, "Default fuel config should work")
    assert_true(sample_battery['rate'] > 0, "Default battery config should work")
    
    print(f"  ✅ Configuration integration test passed")
    print(f"    Sample calculations: {sample_power/1000:.1f}kW, {sample_fuel['rate']:.1f}L/hr, {sample_battery['rate']:.1f}kW")
    return True

# Run integration tests
run_test("Complete Workflow", test_complete_workflow)
run_test("Data Consistency", test_data_consistency)
run_test("Configuration Integration", test_configuration_integration)

## 10. Test Summary {#summary}

In [ ]:
# Display test summary
print("\n" + "="*60)
print("🧪 TEST SUMMARY")
print("="*60)

total_tests = test_results['passed'] + test_results['failed']
pass_rate = (test_results['passed'] / total_tests * 100) if total_tests > 0 else 0

print(f"\n📊 Results:")
print(f"  ✅ Passed: {test_results['passed']}")
print(f"  ❌ Failed: {test_results['failed']}")
print(f"  📈 Pass Rate: {pass_rate:.1f}%")

if test_results['errors']:
    print(f"\n🐛 Errors:")
    for i, error in enumerate(test_results['errors'], 1):
        print(f"  {i}. {error}")

# Overall assessment
if test_results['failed'] == 0:
    print(f"\n🎉 ALL TESTS PASSED! Package is working correctly.")
elif pass_rate >= 80:
    print(f"\n✅ Most tests passed ({pass_rate:.1f}%). Package is mostly functional.")
elif pass_rate >= 50:
    print(f"\n⚠️ Some tests failed ({pass_rate:.1f}%). Package needs attention.")
else:
    print(f"\n❌ Many tests failed ({pass_rate:.1f}%). Package has significant issues.")

print(f"\n🔧 Test Environment:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  Platform: {sys.platform}")
print(f"  Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*60)

## Additional Interactive Testing

Use the cells below for additional interactive testing and exploration:

In [ ]:
# Interactive testing cell - modify as needed
# Example: Test custom scenarios

def custom_test():
    """Add your custom tests here."""
    print("🔧 Custom test area - add your own tests here")
    
    # Example custom test
    try:
        # Your test code here
        result = "Your test result"
        print(f"✅ Custom test result: {result}")
        return True
    except Exception as e:
        print(f"❌ Custom test failed: {e}")
        return False

# Uncomment to run custom test
# run_test("Custom Test", custom_test)

In [ ]:
# Debug cell - use for investigating specific issues
print("🔍 Debug cell - use for investigating specific issues")

# Example: Inspect a specific component
# generator = DrivingCycleGenerator()
# print(f"Generator class: {type(generator)}")
# print(f"Available methods: {[m for m in dir(generator) if not m.startswith('_')]}")

# Example: Test specific parameters
# print(f"Default vehicle params: {DEFAULT_VEHICLE_PARAMS}")
# print(f"Default dynamics params: {DEFAULT_DYNAMICS_PARAMS}")